In [0]:
from pyspark.sql.functions import col, when, sum, round, to_timestamp, to_date, current_timestamp, date_format
from pyspark.sql.types import StringType

In [0]:
df_transacciones = spark.table("finanzas.silver_transacciones")
df_clientes = spark.table("finanzas.silver_clientes")

### Agregar datos para Insights

In [0]:
df_transacciones = (
                    df_transacciones.groupBy("id_cliente", "fecha_transaccion")
                                    .agg(
                                        sum(col("deposito_PEN")).alias("deposito"),
                                        sum(col("retiro_PEN")).alias("retiro")
                                        )
                    )

In [0]:
df = (
        df_transacciones.join(df_clientes, on="id_cliente", how="inner")\
                        .select(
                                "id_cliente", "documento", "nombre_completo", "fecha_transaccion",
                                round(col("deposito").cast("float"), 4).alias("deposito"),
                                round(col("retiro").cast("float"), 4).alias("retiro"),
                                current_timestamp().alias("fecha_carga")
                            )
    )

### Crear y Poblar Tabla Delta

In [0]:
try:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("finanzas.gold_fact_transacciones")
except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error
    error_summary = str(e)
    # Traza del error (ver en que parte se generó el error)
    error_trace = traceback.format_exc()
    
    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full

    dbutils.jobs.taskValues.set(key="error", value=error_msg)
    raise e